#1. Preparing environment

In [ ]:
import torch
from torch import nn
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import shutil
import os
import zipfile
from tqdm.auto import tqdm
from pathlib import Path

In [ ]:
#Setup agnostic code and check versions
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:'{device}', PyTorch version:{torch.__version__}, Torchversion:{torchvision.__version__}")

Device:'cpu', PyTorch version:2.1.0+cu118, Torchversion:0.16.0+cu118


#2. Data preparation
##2.1 Get annotations data

In [ ]:
annotations_path = Path('/content/drive/MyDrive/Colab Notebooks/hx_training_classify/FSW dataset A_annotations.csv')
df_annotations = pd.read_csv(annotations_path, sep=";")
df_annotations

,#,begin,end,Temperatur,Grat,Nahtinneres,temp_label,grat_label,inner_label,filename,img_path,Weld Seam Nr.,Group
0,0,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-25.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-2...,1,1
1,1,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-14.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
2,2,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-37.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-3...,1,1
3,3,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-57.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-5...,1,1
4,4,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-18.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8319,8455,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-52.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8320,8456,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-36.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8321,8457,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-31.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8322,8458,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-44.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7


##2.2 Sort images

In [ ]:
#Create data directory
data_path = Path("./data")
data_path.mkdir(parents=True, exist_ok=True)

In [ ]:
#Copy zip file from drive to data path
drive_zip_path = Path("/content/drive/MyDrive/Colab Notebooks/hx_training_classify/hx_training_classify.zip")
zip_path = Path(shutil.copy(drive_zip_path, data_path))

In [ ]:
#Extract zipfile in data dir
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(data_path)

In [ ]:
#remove original zip file
os.remove(zip_path)

In [ ]:
#Create group directories and label directories
def group_dir_creation(num_groups, target_dir, label_name, label_classes):
  #Checks for Groups directory
  if os.path.exists("./data") and os.path.isdir("./data/Groups"):
    print("Groups dir already exists")
  else:
    groups_path = Path("./data/Groups")
    groups_path.mkdir(parents=True, exist_ok=True)
  #Creates group directories with their respected label groups
  for num in range(1,num_groups + 1):
    if os.path.exists(f"./data/Groups/group_{num}"):
      print(f"Group directory {num} already exists")
    else:
      temp_group_path = Path(f"./data/Groups/group_{num}")
      temp_group_path.mkdir(parents=True, exist_ok=True)

    temp_label_dir = f"./data/Groups/group_{num}/{label_name}"
    if os.path.exists(temp_label_dir):
      print(f"Group directory {temp_label_dir} already exists")
    else:
      temp_label_path = Path(temp_label_dir)
      temp_label_path.mkdir(parents=True, exist_ok=True)

    #Creation of label classes directories
    for class_name in label_classes:
      class_name_path = Path(temp_label_dir + f"/{class_name}")
      #print(class_name_path)
      class_name_path.mkdir(parents=True, exist_ok=True)

In [ ]:
#temp_label directories creation
group_dir_creation(7, "./data/Groups", "temp_label", ["temp_ok_0","temp_kalt_1","temp_heiss_2"])
#grat_label directories creation
group_dir_creation(7, "./data/Groups", "grat_label", ["grat_ok_0","grat_stark_1"])
#inner_label directories creation
group_dir_creation(7, "./data/Groups", "inner_label", ["inner_ok_0","inner_lof_1", "inner_galling_2"])

Groups dir already exists
Group directory 1 already exists
Group directory 2 already exists
Group directory 3 already exists
Group directory 4 already exists
Group directory 5 already exists
Group directory 6 already exists
Group directory 7 already exists
Groups dir already exists
Group directory 1 already exists
Group directory 2 already exists
Group directory 3 already exists
Group directory 4 already exists
Group directory 5 already exists
Group directory 6 already exists
Group directory 7 already exists


In [ ]:
groups_arr = df_annotations["Group"]
print(f"Length of groups_arr: {len(groups_arr)}")
temp_arr = df_annotations["temp_label"]
print(f"Length of groups_arr: {len(temp_arr)}")
print(max(groups_arr))

Length of groups_arr: 8324
Length of groups_arr: 8324
7


In [ ]:
#Convert data frame to dictionary
df_dictionary = df_annotations.to_dict()

In [ ]:
def getKeyPositionsByGroup(data_dictionary):
  groups = max(data_dictionary["Group"].values())
  label_position_dict = {}
  for group in range(1, groups + 1):
    label_position_dict[f"groupPos_{group}"] = {key for key, value in data_dictionary["Group"].items() if value == group}
  return label_position_dict

In [ ]:
positions_by_groups = getKeyPositionsByGroup(df_dictionary)
positions_by_groups.keys()
max(positions_by_groups["groupPos_2"])

1915

In [ ]:
min(positions_by_groups['groupPos_2'])

950

In [ ]:
def add_string_to_file_name(original_path, string_to_add):
    """
    Add a string to the file name before the extension.

    Parameters:
    - original_path (str): The original file path.
    - string_to_add (str): The string to be added to the file name.

    Returns:
    - str: The modified file path with the specified string added before the extension.
    """
    # Split the file name and extension
    file_name, file_extension = os.path.splitext(original_path)

    # Construct the new file path with the added string
    new_file_name = file_name + "_" + string_to_add
    new_path = os.path.join(os.path.dirname(original_path), new_file_name + file_extension)

    return new_path

In [ ]:
def orderImages(origin_path, #./data/hx_training_classify
                dest_path, #./data/Groups
                data_frame, #csv table
                data_dictionary,
                groups_dict, #number or groups in the data_frame
                label): #label we are looking forward to order images to
  label_arr = data_frame[label] #array from desired label
  weld_seam_nr_arr = data_frame["Weld Seam Nr."] #array with weld number
  filenames_arr = data_frame["filename"] #array with filenames in order
  #groups_arr = data_frame["Group"]
  image_path = data_frame["img_path"]
  #print(len(label_positions_1))
  #print(label_positions_1[:10])
  directories_path = {}

  for group in range(1, len(groups_dict.keys()) + 1):
    #1,2,3,4,5,6,7
    #label_dict = data_dictionary[label]
    lowest_bound = min(groups_dict[f"groupPos_{group}"])
    highest_bound = max(groups_dict[f"groupPos_{group}"]) + 1
    #print(f"low: {lowest_bound}, high: {highest_bound}")
    temp_weld_seam_nr_arr = weld_seam_nr_arr[lowest_bound:highest_bound]
    #print(temp_weld_seam_nr_arr)


    print(f"Copying {label} to {dest_path}")
    for i in tqdm(range(lowest_bound,highest_bound)):
      #i stands for the index location of each group
      temp_dir = f"{origin_path}/{temp_weld_seam_nr_arr[i]}/cutouts/{filenames_arr[i]}"
      temp_dir_path = Path(temp_dir)
      new_file_name = add_string_to_file_name(filenames_arr[i], str(temp_weld_seam_nr_arr[i]))
      #print(temp_dir_path) #data/hx_training_classify/1/cutouts/region-2-25.JPG
      #print(new_file_name) #region-2-25_1.JPG
      directories = os.listdir(dest_path + f"/group_{group}/{label}")
      value_label_arr = label_arr[i] #0,1,2
      #print(value_label_arr)
      j = 0
      #print(len(directories))
      while j < len(directories):
        directories_path[f"group_{group}_{label}_{directories[j]}"] = {"PATH": Path(str(dest_path)+ f"/group_{group}/{label}/{directories[j]}")}
        if value_label_arr == j:
          NEW_DEST_FILE_NAME_PATH = Path(str(dest_path)+ f"/group_{group}/{label}/{directories[j]}/" + new_file_name)
          #print(NEW_DEST_FILE_NAME_PATH) #data/Groups/group_1/temp_label/temp_ok_0/region-2-25_1.JPG
          shutil.copy2(temp_dir_path, NEW_DEST_FILE_NAME_PATH)

          break
        else:
          j += 1
  return directories_path

In [ ]:
temp_label_directories_path = orderImages(origin_path="./data/hx_training_classify",
            dest_path="./data/Groups",
            data_frame = df_annotations,
            data_dictionary=df_dictionary,
            groups_dict= positions_by_groups,
            label="temp_label")
grat_label_directories_path = orderImages(origin_path="./data/hx_training_classify",
            dest_path="./data/Groups",
            data_frame = df_annotations,
            data_dictionary=df_dictionary,
            groups_dict= positions_by_groups,
            label="grat_label")
inner_label_directories_path = orderImages(origin_path="./data/hx_training_classify",
            dest_path="./data/Groups",
            data_frame = df_annotations,
            data_dictionary=df_dictionary,
            groups_dict= positions_by_groups,
            label="inner_label")

Copying temp_label to ./data/Groups


  0%|          | 0/950 [00:00<?, ?it/s]

Copying temp_label to ./data/Groups


  0%|          | 0/966 [00:00<?, ?it/s]

Copying temp_label to ./data/Groups


...

  0%|          | 0/1066 [00:00<?, ?it/s]

Copying inner_label to ./data/Groups


  0%|          | 0/1522 [00:00<?, ?it/s]

Copying inner_label to ./data/Groups


  0%|          | 0/1430 [00:00<?, ?it/s]

In [ ]:
inner_label_directories_path

{'group_1_inner_label_inner_ok_0': {'PATH': PosixPath('data/Groups/group_1/inner_label/inner_ok_0')},
 'group_1_inner_label_inner_lof_1': {'PATH': PosixPath('data/Groups/group_1/inner_label/inner_lof_1')},
 'group_2_inner_label_inner_ok_0': {'PATH': PosixPath('data/Groups/group_2/inner_label/inner_ok_0')},
 'group_2_inner_label_inner_lof_1': {'PATH': PosixPath('data/Groups/group_2/inner_label/inner_lof_1')},
 'group_2_inner_label_inner_galling_2': {'PATH': PosixPath('data/Groups/group_2/inner_label/inner_galling_2')},
 'group_3_inner_label_inner_ok_0': {'PATH': PosixPath('data/Groups/group_3/inner_label/inner_ok_0')},
 'group_3_inner_label_inner_lof_1': {'PATH': PosixPath('data/Groups/group_3/inner_label/inner_lof_1')},
 'group_3_inner_label_inner_galling_2': {'PATH': PosixPath('data/Groups/group_3/inner_label/inner_galling_2')},
 'group_4_inner_label_inner_ok_0': {'PATH': PosixPath('data/Groups/group_4/inner_label/inner_ok_0')},
 'group_5_inner_label_inner_ok_0': {'PATH': PosixPath('d

#3 Create `Dataset`s and `DataLoader`s
##3.1 Create `Dataset`s

In [ ]:
def datasetCreator(target_dir, height=64, width=64):
  data_transform = transforms.Compose([
    transforms.Resize((height, width)),
    transforms.ToTensor()
  ])

In [ ]:
def createDatasetsInDirectory(origin_dir, label):
  outer_directories = os.listdir(origin_dir)
  for outer_dir in outer_directories:
    #print(dir)
    inner_directory_path = os.path.join(origin_dir, outer_dir)
    inner_directory = os.listdir(inner_directory_path)
    for inner_dir in inner_directory:
      label_directory_path = os.path.join(inner_directory_path, label)
      label_directories = os.listdir(label_directory_path)
      for label_dir in label_directories:



createDatasetsInDirectory("./data/Groups", "temp_label")

temp_label_temp_ok_0
temp_label_temp_kalt_1
temp_label_temp_heiss_2
inner_label_temp_ok_0
inner_label_temp_kalt_1
inner_label_temp_heiss_2
grat_label_temp_ok_0
grat_label_temp_kalt_1
grat_label_temp_heiss_2
temp_label_temp_ok_0
temp_label_temp_kalt_1
temp_label_temp_heiss_2
inner_label_temp_ok_0
inner_label_temp_kalt_1
inner_label_temp_heiss_2
grat_label_temp_ok_0
grat_label_temp_kalt_1
grat_label_temp_heiss_2
temp_label_temp_ok_0
temp_label_temp_kalt_1
temp_label_temp_heiss_2
inner_label_temp_ok_0
inner_label_temp_kalt_1
inner_label_temp_heiss_2
grat_label_temp_ok_0
grat_label_temp_kalt_1
grat_label_temp_heiss_2
temp_label_temp_ok_0
temp_label_temp_kalt_1
temp_label_temp_heiss_2
inner_label_temp_ok_0
inner_label_temp_kalt_1
inner_label_temp_heiss_2
grat_label_temp_ok_0
grat_label_temp_kalt_1
grat_label_temp_heiss_2
temp_label_temp_ok_0
temp_label_temp_kalt_1
temp_label_temp_heiss_2
inner_label_temp_ok_0
inner_label_temp_kalt_1
inner_label_temp_heiss_2
grat_label_temp_ok_0
grat_label_t